In [4]:
import pandas as pd
data = pd.read_csv("heart.csv")
print(data.head())
print("\n Dataset Shape:", data.shape)
print("\nColumn Information:")
print(data.info())
print("\nClass Distribution (target column):")
print(data['target'].value_counts())

data.describe()


   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

 Dataset Shape: (1025, 14)

Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null  

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.00000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,54.434146,0.695610,0.942439,131.611707,246.00000,0.149268,0.529756,149.114146,0.336585,1.071512,1.385366,0.754146,2.323902,0.513171
std,9.072290,0.460373,1.029641,17.516718,51.59251,0.356527,0.527878,23.005724,0.472772,1.175053,0.617755,1.030798,0.620660,0.500070
min,29.000000,0.000000,0.000000,94.000000,126.00000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.000000,0.000000,0.000000,120.000000,211.00000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,56.000000,1.000000,1.000000,130.000000,240.00000,0.000000,1.000000,152.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,275.00000,0.000000,1.000000,166.000000,1.000000,1.800000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.00000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [5]:
import numpy as np
print("\n Missing values per column:")
print(data.isnull().sum())

# Fill missing numerical values with mean
for col in data.select_dtypes(include=[np.number]).columns:
    data[col] = data[col].fillna(data[col].mean())


 Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
X = data.drop('target', axis=1)
y = data['target']

# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("\n Dataset shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")


 Dataset shapes:
X_train: (717, 13), y_train: (717,)
X_val: (154, 13), y_val: (154,)
X_test: (154, 13), y_test: (154,)


In [7]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit scaler on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test sets using the same scaler
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [8]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on validation set
y_val_pred = rf_model.predict(X_val_scaled)

# Metrics
val_accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", val_accuracy)
print("\nClassification Report (Validation Set):\n", classification_report(y_val, y_val_pred))



Validation Accuracy: 0.974025974025974

Classification Report (Validation Set):
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        92
           1       1.00      0.94      0.97        62

    accuracy                           0.97       154
   macro avg       0.98      0.97      0.97       154
weighted avg       0.98      0.97      0.97       154



In [10]:
# Predict on test set
y_test_pred = rf_model.predict(X_test_scaled)

# Metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_accuracy)
print("\nClassification Report (Test Set):\n", classification_report(y_test, y_test_pred))


Test Accuracy: 0.987012987012987

Classification Report (Test Set):
               precision    recall  f1-score   support

           0       0.97      1.00      0.99        67
           1       1.00      0.98      0.99        87

    accuracy                           0.99       154
   macro avg       0.99      0.99      0.99       154
weighted avg       0.99      0.99      0.99       154



In [11]:
# =========================
# 1. Import Libraries
# =========================
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# =========================
# 2. Scale the Data
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# =========================
# 3. Random Forest with GridSearch
# =========================
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rf_model = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(rf_model, rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_rf.fit(X_train_scaled, y_train)
best_rf = grid_search_rf.best_estimator_

# Predictions
y_val_rf = best_rf.predict(X_val_scaled)
y_test_rf = best_rf.predict(X_test_scaled)

# =========================
# 4. Logistic Regression
# =========================
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_val_log = log_model.predict(X_val_scaled)
y_test_log = log_model.predict(X_test_scaled)

# =========================
# 5. Neural Network (MLP)
# =========================
nn_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
nn_model.fit(X_train_scaled, y_train)

y_val_nn = nn_model.predict(X_val_scaled)
y_test_nn = nn_model.predict(X_test_scaled)

# =========================
# 6. Print Metrics
# =========================
models = ['Random Forest', 'Logistic Regression', 'Neural Network']
val_accuracies = [accuracy_score(y_val, y_val_rf),
                  accuracy_score(y_val, y_val_log),
                  accuracy_score(y_val, y_val_nn)]
test_accuracies = [accuracy_score(y_test, y_test_rf),
                   accuracy_score(y_test, y_test_log),
                   accuracy_score(y_test, y_test_nn)]

comparison_df = pd.DataFrame({
    'Model': models,
    'Validation Accuracy': val_accuracies,
    'Test Accuracy': test_accuracies
})

print("=== Model Comparison ===")
print(comparison_df)

# =========================
# 7. Detailed Classification Reports (Optional)
# =========================
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_test_rf))

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_test_log))

print("\n--- Neural Network Classification Report ---")
print(classification_report(y_test, y_test_nn))


Fitting 5 folds for each of 48 candidates, totalling 240 fits
=== Model Comparison ===
                 Model  Validation Accuracy  Test Accuracy
0        Random Forest             0.987013       0.993506
1  Logistic Regression             0.818182       0.792208
2       Neural Network             0.974026       0.987013

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        67
           1       1.00      0.99      0.99        87

    accuracy                           0.99       154
   macro avg       0.99      0.99      0.99       154
weighted avg       0.99      0.99      0.99       154


--- Logistic Regression Classification Report ---
              precision    recall  f1-score   support

           0       0.79      0.72      0.75        67
           1       0.80      0.85      0.82        87

    accuracy                           0.79       154
   macro avg       0.79      0.78   

In [12]:
import joblib

# Save the best model and scaler
joblib.dump(best_rf, "heart_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("✅ Model and Scaler saved successfully!")


✅ Model and Scaler saved successfully!


In [15]:
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression', 'Neural Network'],
    'Test Accuracy': [rf_acc, log_acc, nn_acc]
})

print(comparison_df)

# Optional: Detailed report for best model
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf))


                 Model  Test Accuracy
0        Random Forest       0.985366
1  Logistic Regression       0.790244
2       Neural Network       0.829268

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       102
           1       1.00      0.97      0.99       103

    accuracy                           0.99       205
   macro avg       0.99      0.99      0.99       205
weighted avg       0.99      0.99      0.99       205

